# Coarse answer plus LoFTR evidence below the bar

**Question.** Iteration 25/25b: the coarse peak alone is not safe (z gate does not transfer across the new, official and NUDT sets). Does a second, independent check separate right coarse answers from wrong ones on all three sets? The check: LoFTR + RANSAC inliers at the coarse peak (counted even below the bar of 8), and whether LoFTR's best place agrees with the coarse peak.

**What it records.** For every photo, the 6 standard queries; per query the first M=5 places in coarse order (DINO optional with `CE_DINO=1`), the RANSAC inliers of each place and the error of each place in sigma. Join with `queries_<TAG>.csv` of Coarse_Confidence for the coarse z.

**Kill rule (written before the run).** A gate (coarse z and/or inliers at the coarse peak, and/or agreement) is useful only if on all three sets it gives 0 answers beyond 8 sigma among the photos it answers, and it answers at least 40 % of the photos the pipeline skips above 800 m on the new set. A threshold chosen on one set and tested on the others only counts if it holds on the others unchanged.

Environment: `CE_HEADLESS`, `CE_DATASET`, `CE_STORE`, `CE_LEVELS`, `CE_TAG`, `CE_FIRST`, `CE_LIMIT`, `CE_JITTER`, `CE_OUT`, `CE_GEOMETRY`, `CE_MODELS`, `CE_DINO`.

In [ ]:
import os, sys, math
import numpy as np, pandas as pd, torch
HEADLESS = os.environ.get("CE_HEADLESS", "../../../../Main_Pipeline_Headless")
sys.path.insert(0, os.path.abspath(HEADLESS))
import settings
settings.LEVELS = [int(v) for v in os.environ.get("CE_LEVELS", "3,4,5,6,7").split(",")]
settings.MODELS_DIR = os.environ.get("CE_MODELS", os.path.join(os.path.abspath(HEADLESS), "models"))
settings.USE_DINO = os.environ.get("CE_DINO", "0") == "1"
for item in os.environ.get("CE_GEOMETRY", "").split(","):
    if item:
        key, value = item.split("="); settings.GEOMETRY[key] = (value == "True") if value in ("True", "False") else float(value)
print("geometry", settings.GEOMETRY, "dino", settings.USE_DINO)
from geoloc import photo, queries, scoring
from geoloc.cli import read_rows, move_hint
from geoloc.levels import level_for_height
from geoloc.pipeline import Localiser

DATASET = os.environ["CE_DATASET"]; STORE = os.environ["CE_STORE"]
TAG = os.environ.get("CE_TAG", "run"); FIRST = int(os.environ.get("CE_FIRST", 0)); LIMIT = int(os.environ.get("CE_LIMIT", 0))
JITTER = float(os.environ.get("CE_JITTER", 5000)); OUT = os.environ.get("CE_OUT", "results/coarse_evidence")
os.makedirs(OUT, exist_ok=True)
rows = read_rows(DATASET)[FIRST:]
if LIMIT: rows = rows[:LIMIT]
localiser = Localiser(STORE, settings.MODELS_DIR); localiser.warm_up()
store = localiser.store
print(len(rows), "photos"); 

In [ ]:
records = []
query_list = queries.query_list()
for index, meta in enumerate(rows):
    meta = move_hint(meta, index + FIRST, JITTER, 0) if JITTER > 0 else meta
    camera = photo.read_camera(meta); pose = photo.read_pose(meta)
    prior = store.gps_to_metres(float(meta["lat_sc"]), float(meta["lon_sc"]))
    truth = store.gps_to_metres(float(meta["lat_gtruth"]), float(meta["lon_gtruth"]))
    sigma = scoring.sigma_m(pose["height"], camera["fx"])
    level = level_for_height(pose["height"], camera)
    picture = photo.prepare_photo(os.path.join(DATASET, "images", meta["filename"]), pose, camera, level)
    cache = {}
    for q, query in enumerate(query_list):
        try:
            result = localiser.run_query(picture, pose, camera, level, prior, query, {}, cache)
        except ValueError:
            continue
        peaks = result["peaks"]; inliers = result["stage2"]["inliers"]; shifts = result["stage2"]["shifts"]
        row = dict(photo=index + FIRST, name=meta["filename"], q=q, height=pose["height"], sigma_m=sigma)
        best = int(np.argmax(inliers))
        for k in range(len(peaks)):
            row["x%d" % k] = peaks[k, 1]; row["y%d" % k] = peaks[k, 2]
            row["inl%d" % k] = inliers[k]
            row["err%d" % k] = math.hypot(peaks[k, 1] - truth[0], peaks[k, 2] - truth[1]) / sigma
        row["best_k"] = best; row["best_inl"] = inliers[best]
        row["err_best_shifted"] = math.hypot(peaks[best, 1] + shifts[best, 0] - truth[0], peaks[best, 2] + shifts[best, 1] - truth[1]) / sigma
        records.append(row)
    if index % 20 == 0:
        print(index + 1, "/", len(rows), flush=True)
        pd.DataFrame(records).to_csv(os.path.join(OUT, "evidence_%s.csv" % TAG), index=False)
table = pd.DataFrame(records)
table.to_csv(os.path.join(OUT, "evidence_%s.csv" % TAG), index=False)
print(len(table), "query rows saved")

In [ ]:
# quick look: inliers at the first (coarse) place for right and wrong first places
right = table.err0 <= 2
print("queries:", len(table), " first place right:", round(right.mean(), 2))
print("inliers at first place, right vs wrong:")
print(table.groupby(right).inl0.describe().round(1))